# CMPE 460 BiLSTM Training on SecureAI-SE

**Member 2**

This notebook trains a bidirectional LSTM classifier using the SecureAI-SE dataset revision and character encoding used in `03_textcnn_secureai.ipynb`. Run all cells from top to bottom.

The model uses 32-dimensional character embeddings and one bidirectional LSTM layer with 64 hidden units per direction. Dropout of 0.3 is applied before the classification layer.

The experiment uses seed 42, batch size 64, Adam with a learning rate of 0.001, at most 15 epochs, and early stopping after four epochs without an improvement in validation macro-F1.


## 1. Import Packages and Load the Dataset

The cell prints each setup stage. It uses the pinned Parquet files from the Hugging Face cache when available and downloads missing files. Package installation is performed once in your Python environment. If an import fails, run the following command in a separate code cell, restart the kernel, and run this notebook again.

```python
%pip install torch datasets scikit-learn pandas numpy pyarrow huggingface-hub
```

A CUDA-enabled PyTorch installation uses an available NVIDIA GPU. A CPU installation runs the model on the CPU.


In [ ]:
from time import perf_counter

setup_started = perf_counter()
print("Step 1/3: importing packages...", flush=True)

import random

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from datasets import Features
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import LocalEntryNotFoundError
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print(f"Imports completed in {perf_counter() - setup_started:.1f}s.", flush=True)
print("Step 2/3: loading dataset files...", flush=True)

DATASET_ID = "SecureAI-SE/http-attack-requests"
DATASET_REVISION = "9ada662402e0620cc4e3bd32b2cc3c2025bb3804"
SPLITS = ("train", "validation", "test")
MAX_LENGTH = 256
PAD_ID = 0
UNK_ID = 1

frames = {}
label_names = []

for split in SPLITS:
    download_options = {
        "repo_id": DATASET_ID,
        "repo_type": "dataset",
        "revision": DATASET_REVISION,
        "filename": f"data/{split}-00000-of-00001.parquet",
    }
    try:
        parquet_path = hf_hub_download(**download_options, local_files_only=True)
        print(f"{split}: reading cached file.", flush=True)
    except LocalEntryNotFoundError:
        print(f"{split}: downloading dataset file...", flush=True)
        parquet_path = hf_hub_download(**download_options)

    table = pq.read_table(parquet_path)
    split_label_names = Features.from_arrow_schema(table.schema)["label"].names
    if split == "train":
        label_names = split_label_names
    assert split_label_names == label_names
    frames[split] = table.to_pandas()[["request", "label"]].copy()

print("Step 3/3: encoding requests...", flush=True)

train_characters = sorted(set("".join(frames["train"]["request"])))
char_to_id = {
    char: index
    for index, char in enumerate(train_characters, start=2)
}


def encode_requests(requests):
    encoded = np.full((len(requests), MAX_LENGTH), PAD_ID, dtype=np.int64)
    for row_index, text in enumerate(requests):
        character_ids = [
            char_to_id.get(char, UNK_ID)
            for char in text[:MAX_LENGTH]
        ]
        encoded[row_index, :len(character_ids)] = character_ids
    return encoded


X = {
    split: encode_requests(frames[split]["request"].tolist())
    for split in SPLITS
}
y = {
    split: frames[split]["label"].to_numpy(dtype=np.int64)
    for split in SPLITS
}

for split in SPLITS:
    assert X[split].shape == (len(y[split]), MAX_LENGTH)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < len(char_to_id) + 2
    print(f"{split}: X={X[split].shape}, y={y[split].shape}", flush=True)

print(f"Data setup completed in {perf_counter() - setup_started:.1f}s.", flush=True)


Step 1/3: importing packages...


Imports completed in 5.0s.


Step 2/3: loading dataset files...


train: reading cached file.


validation: reading cached file.


test: reading cached file.


Step 3/3: encoding requests...


train: X=(4900, 256), y=(4900,)


validation: X=(700, 256), y=(700,)


test: X=(1400, 256), y=(1400,)


Data setup completed in 5.1s.


## 2. Configure Training and Create Data Loaders

The seed, embedding size, batch size, learning rate, maximum epochs, and early-stopping patience match the TextCNN experiment. Each training batch is shuffled using the recorded seed.

The CPU thread count is set to two after a short timing comparison on the development machine. The actual device, thread count, PyTorch version, and epoch durations are printed and should accompany any runtime comparison.


In [ ]:
SEED = 42
BATCH_SIZE = 64
EMBEDDING_DIM = 32
HIDDEN_SIZE = 64
CPU_THREADS = 2
DROPOUT = 0.3
LEARNING_RATE = 0.001
MAX_EPOCHS = 15
PATIENCE = 4
LOG_EVERY_BATCHES = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(CPU_THREADS)
vocabulary_size = len(char_to_id) + 2
number_of_classes = len(label_names)

loaders = {}
for split in ("train", "validation"):
    samples = TensorDataset(
        torch.from_numpy(X[split]),
        torch.from_numpy(y[split]),
    )
    loaders[split] = DataLoader(
        samples,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

print(f"Device: {device}", flush=True)
print(f"PyTorch version: {torch.__version__}", flush=True)
print(f"CPU threads: {torch.get_num_threads()}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
else:
    print("This kernel will train on the CPU.", flush=True)
print(f"Dataset revision: {DATASET_REVISION}", flush=True)
print(f"Vocabulary size: {vocabulary_size}", flush=True)
print(f"Number of classes: {number_of_classes}", flush=True)
print(f"Maximum input length: {MAX_LENGTH}", flush=True)
print(f"Hidden units per direction: {HIDDEN_SIZE}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Learning rate: {LEARNING_RATE}", flush=True)
print(f"Maximum epochs: {MAX_EPOCHS}", flush=True)
print(f"Early-stopping patience: {PATIENCE}", flush=True)


Device: cpu


PyTorch version: 2.15.0+cpu


CPU threads: 2


This kernel will train on the CPU.


Dataset revision: 9ada662402e0620cc4e3bd32b2cc3c2025bb3804


Vocabulary size: 73


... (3 outputs left out)

Batch size: 64


Learning rate: 0.001


Maximum epochs: 15


Early-stopping patience: 4


## 3. Define the BiLSTM

Each character ID is mapped to a learned embedding. A bidirectional LSTM processes the character sequence in both directions. The final forward and backward hidden states are concatenated and passed through dropout and a linear layer to produce seven class scores.

The sequence lengths count real characters after truncation. Packing the sequences keeps padding outside the recurrent computation. The classifier uses the final hidden state of each direction.

[PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) · [Packed sequences](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html)


In [ ]:
class BiLSTM(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        number_of_classes,
        embedding_dim=32,
        hidden_size=64,
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_id,
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, number_of_classes)

    def forward(self, tokens):
        lengths = tokens.ne(self.pad_id).sum(dim=1).cpu()
        embedded = self.embedding(tokens)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden_states, _) = self.lstm(packed)
        combined = torch.cat(
            [hidden_states[-2], hidden_states[-1]],
            dim=1,
        )
        return self.classifier(self.dropout(combined))


model = BiLSTM(
    vocabulary_size=vocabulary_size,
    number_of_classes=number_of_classes,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    pad_id=PAD_ID,
).to(device)

for split in ("train", "validation"):
    assert (X[split] != PAD_ID).sum(axis=1).min() > 0

model.eval()
with torch.no_grad():
    example_inputs = torch.from_numpy(X["train"][:BATCH_SIZE]).to(device)
    example_scores = model(example_inputs)

assert example_scores.shape == (len(example_inputs), number_of_classes)
assert torch.isfinite(example_scores).all()

print(model)
print(f"Example output shape: {tuple(example_scores.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


BiLSTM(
  (embedding): Embedding(73, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=7, bias=True)
)
Example output shape: (64, 7)
Trainable parameters: 53,415


## 4. Train and Select the Best Epoch

Cross-entropy compares the class scores with the integer labels. Training progress is printed after the first batch, every 20 batches, and the final batch of each epoch. The first batch may take longer while PyTorch initializes its execution kernels.

After each epoch, validation accuracy and macro-F1 are calculated. The best weights are kept in memory. Training stops after four consecutive epochs without an improvement in validation macro-F1. Epoch durations are recorded alongside the metrics.


In [ ]:
from copy import deepcopy

from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        scores = model(inputs)
        loss = criterion(scores, targets)

        total_loss += loss.item() * targets.size(0)
        true_labels.extend(targets.cpu().tolist())
        predicted_labels.extend(scores.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            labels=list(range(number_of_classes)),
            average="macro",
            zero_division=0,
        ),
    }
    return metrics, true_labels, predicted_labels


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history_rows = []

print(f"Starting training for up to {MAX_EPOCHS} epochs on {device}.", flush=True)

training_started = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = perf_counter()
    model.train()
    total_training_loss = 0.0

    for batch_index, (inputs, targets) in enumerate(loaders["train"], start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(inputs)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        total_training_loss += loss.item() * targets.size(0)

        if (
            batch_index == 1
            or batch_index % LOG_EVERY_BATCHES == 0
            or batch_index == len(loaders["train"])
        ):
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
                f"batch {batch_index:02d}/{len(loaders['train'])} | "
                f"elapsed {perf_counter() - epoch_started:.1f}s",
                flush=True,
            )

    training_loss = total_training_loss / len(loaders["train"].dataset)
    validation, _, _ = evaluate_model(model, loaders["validation"])
    epoch_seconds = perf_counter() - epoch_started

    history_rows.append({
        "epoch": epoch,
        "train_loss": training_loss,
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["accuracy"],
        "validation_macro_f1": validation["macro_f1"],
        "epoch_seconds": epoch_seconds,
    })

    print(
        f"Epoch {epoch:02d} | train loss {training_loss:.4f} | "
        f"val loss {validation['loss']:.4f} | "
        f"val accuracy {validation['accuracy']:.4f} | "
        f"val macro-F1 {validation['macro_f1']:.4f} | "
        f"time {epoch_seconds:.1f}s",
        flush=True,
    )

    if validation["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation["macro_f1"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}.")
        break

model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(history_rows)

print(f"Restored the best epoch: {best_epoch}")
print(f"Total training time: {perf_counter() - training_started:.1f}s")
display(history.round(4))


Starting training for up to 15 epochs on cpu.


Epoch 01/15 | batch 01/77 | elapsed 0.5s


Epoch 01/15 | batch 20/77 | elapsed 11.2s


Epoch 01/15 | batch 40/77 | elapsed 21.9s


Epoch 01/15 | batch 60/77 | elapsed 31.9s


Epoch 01/15 | batch 77/77 | elapsed 40.4s


... (83 outputs left out)

Epoch 15/15 | batch 77/77 | elapsed 187.8s


Epoch 15 | train loss 0.2067 | val loss 0.1462 | val accuracy 0.9529 | val macro-F1 0.9534 | time 189.1s


Restored the best epoch: 15
Total training time: 2020.7s


,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1,epoch_seconds
0,1,1.4891,0.8421,0.7657,0.7652,40.8026
1,2,0.6473,0.4424,0.8614,0.8621,38.2513
2,3,0.4120,0.3103,0.9014,0.9010,38.2908
3,4,0.3081,0.2384,0.9200,0.9198,179.1748
4,5,0.2615,0.3123,0.8929,0.8904,134.9925
5,6,0.2349,0.1905,0.9300,0.9302,190.9054
6,7,0.2403,0.2372,0.9229,0.9235,149.3264
7,8,0.2191,0.1889,0.9357,0.9359,37.6580
8,9,0.1847,0.1750,0.9314,0.9318,180.5672
9,10,0.1814,0.1723,0.9386,0.9383,189.2197


## 5. Report Validation Results

The following metrics describe the model selected using validation macro-F1. Record them as development results in the architecture comparison. Test metrics will be reported after the experiment settings have been selected.

Save this notebook with its outputs after the run completes. Run all cells again to start a new experiment from the recorded seed.


In [ ]:
from sklearn.metrics import classification_report

validation_metrics, validation_labels, validation_predictions = evaluate_model(
    model,
    loaders["validation"],
)

print(f"Best epoch: {best_epoch}")
print(f"Validation accuracy: {validation_metrics['accuracy']:.4f}")
print(f"Validation macro-F1: {validation_metrics['macro_f1']:.4f}")
print()
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=list(range(number_of_classes)),
    target_names=label_names,
    digits=4,
    zero_division=0,
))


Best epoch: 15
Validation accuracy: 0.9529
Validation macro-F1: 0.9534

              precision    recall  f1-score   support

      normal     0.9397    0.9646    0.9520       113
        sqli     0.9681    0.9891    0.9785        92
         xss     0.9783    0.9000    0.9375       100
        ssrf     0.9892    1.0000    0.9946        92
        ssti     0.9903    1.0000    0.9951       102
         lfi     0.8900    0.9271    0.9082        96
   traversal     0.9216    0.8952    0.9082       105

    accuracy                         0.9529       700
   macro avg     0.9539    0.9537    0.9534       700
weighted avg     0.9533    0.9529    0.9527       700

